In [ ]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np

#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

2025-08-12 13:17:54.219812: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-12 13:17:54.224002: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-08-12 13:17:54.224020: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [ ]:

def from_ortho(inp:scm.ortho)->scm.Bounds:
    """
    Takes an ortho object and abstracts out its bounds.
    Aggregation function: will hang if ortho is not done fitting yet. 
    """
    
    ret=scm.Bounds()

    #get the min & max nb parameters across all models
    mins=[]
    maxes=[]
    
    #for each model class
    for var in ["by_cell_type_parameters","by_cre_parameters"]:
        #nb min & max
        for key in getattr(inp,var).nb:
            current=getattr(inp,var).nb[key].result()
            maxes.append(float(current.max()))
            mins.append(float(current.min()))
        
        thetas=[]
        for key in getattr(inp,var).theta:
            current=getattr(inp,var).theta[key].result()
            thetas.append(current)
        
        #we could munge the strings & use setattr but i think this is more readable
        if var=="by_cell_type_parameters":
            ret.by_cell_type_theta=np.mean(thetas)
    
    #min & max nb and add to the return object
    ret.min_mpra_umi=min(mins)
    ret.max_mpra_umi=max(maxes)

    
            

    #by_cre_theta:float
    #by_cell_type_theta:float
    #mean_mpra_barcodes_per_cre:float
    #stdev_mpra_barcodes_per_cre:float

    return ret

def description_from_bounds(inp:scm.Bounds,
                            type="simple_spread",
                            nb_override=None):
    """
    Returns a primordial description dask dataframe

    and a hypothesis set describing the (unimplemented)
    
    Only type implemented for now is `simple_spread`
    - tiled CREs over ...
    other will be 
    - `override` : allowing you to specify exactly what nb param
    """
    #how to 
    pass

def simple_spread():
    """
    Takes 
    """
    pass

In [3]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='8GB')
client=Client(cluster)

In [4]:
xyz=scm.ortho.load(client,".","test_save")

In [13]:
b=from_ortho(xyz)

In [14]:
b

Bounds(metadata=None, min_mpra_umi=1.980090366347442, max_mpra_umi=222.04230199480094, by_cre_theta=None, by_cell_type_theta=3.2159488)

In [5]:
dat=scm.scMPRA_data.from_tsv("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres_v3.tsv")
dat.ortho_filter()
dat.set_negative_controls(["nobody","weak"])
dat.set_reference_cell("liver")
primordial=scm.ortho()
primordial.criss_cross(client,dat=dat)
primordial.extract_params(client)

scMPRAforge: INFO: Dropped 0 of 27 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 0 of 27 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [6]:
primordial.save(".","test_save")

3.2567084
3.396007
3.3175333
3.2008197
2.1678116
3.2843478
3.2969668
3.3159997


In [11]:
xyz.training_data.data["umis_mpra_bc"]

0         74
1        255
2        101
3         27
4        101
        ... 
41966      6
41967      0
41968      2
41969      4
41970      0
Name: umis_mpra_bc, Length: 41971, dtype: int64

In [21]:
min(mins)

2.0135518849371543

In [25]:
cluster.close()